# Teacher pseudo-labeling

Notebook для запуска Teacher inference в Kaggle/GPU-среде. Перед запуском подключите Kaggle Dataset с архивом `nuscenes_pedestrian_crops_filtered.zip` и репозиторий проекта.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path('/kaggle/working/pose-estimation')
DATASET_ZIP = Path('/kaggle/input/nuscenes-pedestrian-crops-filtered/nuscenes_pedestrian_crops_filtered.zip')
DATA_DIR = Path('/kaggle/working/data/processed')
FILTERED_CROPS_DIR = DATA_DIR / 'nuscenes_pedestrian_crops_filtered'
TEACHER_OUTPUT_DIR = Path('/kaggle/working/data/pseudo/nuscenes_pose_teacher')
KEYPOINT_QA_DIR = Path('/kaggle/working/data/qa/keypoints_preview')
MMPOSE_CKPT_DIR = Path('/kaggle/working/mmpose_checkpoints')

In [ ]:
# Если репозиторий не добавлен в Kaggle как Dataset, раскомментируйте clone.
# !git clone <your-repo-url> {PROJECT_DIR}
%cd {PROJECT_DIR}

In [ ]:
# Установка MMPose. В Kaggle версии CUDA/PyTorch могут меняться, поэтому при проблемах
# сверяйтесь с официальной инструкцией MMPose/OpenMMLab.
!pip install -U openmim
!mim install mmengine
!mim install "mmcv>=2.0.1"
!pip install mmpose

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
!unzip -q -o {DATASET_ZIP} -d {DATA_DIR}
!ls -lah {FILTERED_CROPS_DIR}
!head -1 {FILTERED_CROPS_DIR / 'manifest.jsonl'}

## Teacher model

Практичный первый Teacher: RTMPose-l AIC+COCO. Для более сильного Teacher можно заменить config/checkpoint на ViTPose-H, если хватает GPU memory.

In [ ]:
MMPOSE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
!mim download mmpose --config rtmpose-l_8xb256-420e_aic-coco-256x192 --dest {MMPOSE_CKPT_DIR}

MMPOSE_CONFIG = str(MMPOSE_CKPT_DIR / 'rtmpose-l_8xb256-420e_aic-coco-256x192.py')
CHECKPOINT = str(MMPOSE_CKPT_DIR / 'rtmpose-l_simcc-aic-coco_pt-aic-coco_420e-256x192-f016ffe0_20230126.pth')
DEVICE = 'cuda:0'

print(MMPOSE_CONFIG)
print(CHECKPOINT)

## Smoke inference

Сначала запускаем Teacher на маленьком subset. После этого нужно открыть `keypoint_grid.jpg` и глазами проверить качество keypoints.

In [ ]:
!python scripts/run_teacher_inference.py \
  --dataset-dir {FILTERED_CROPS_DIR} \
  --output-dir {TEACHER_OUTPUT_DIR}_smoke \
  --mmpose-config {MMPOSE_CONFIG} \
  --checkpoint {CHECKPOINT} \
  --device {DEVICE} \
  --limit 128

In [ ]:
!python scripts/visualize_keypoints.py \
  --dataset-dir {FILTERED_CROPS_DIR} \
  --labels-path {TEACHER_OUTPUT_DIR}_smoke/pseudo_labels.jsonl \
  --output-dir {KEYPOINT_QA_DIR}_smoke \
  --num-samples 64

from IPython.display import Image, display
display(Image(filename=str(KEYPOINT_QA_DIR) + '_smoke/keypoint_grid.jpg'))

## Full inference

Запускайте только после визуальной проверки smoke-разметки.

In [ ]:
# !python scripts/run_teacher_inference.py \
#   --dataset-dir {FILTERED_CROPS_DIR} \
#   --output-dir {TEACHER_OUTPUT_DIR} \
#   --mmpose-config {MMPOSE_CONFIG} \
#   --checkpoint {CHECKPOINT} \
#   --device {DEVICE}

In [ ]:
# !python scripts/build_coco_pose_dataset.py \
#   --dataset-dir {FILTERED_CROPS_DIR} \
#   --labels-path {TEACHER_OUTPUT_DIR}/pseudo_labels.jsonl \
#   --output-dir /kaggle/working/data/pseudo/nuscenes_pose_coco \
#   --copy-images